**Transkun** — це відкрита нейронна модель (2024 року) для автоматичної транскрипції гри на піаніно з аудіо (MP3, WAV) в MIDI-файл. Вона спеціально створена для піаніно і вміє розпізнавати:

окремі ноти та поліфонію (кілька нот одночасно),
тривалість нот,
гучність (velocity),
динаміку та виразність виконання (timing deviations, педаль).

На відміну від старіших моделей (як madmom), transkun використовує сучасну архітектуру нейронний semi-Markov Conditional Random Field (semi-CRF) — це дозволяє їй "розуміти" послідовність нот і уникати типових помилок (дублі, зайві ноти, пропуски).

In [ ]:

!pip3 install transkun


import os
from google.colab import files
import zipfile


def removeExtension(file):
    return ".".join(os.path.basename(file).split('.')[:-1])

def transcribe(file, outfolder='.'):
  print(file)

  path = os.path.join(outfolder, removeExtension(file)+".mid")
  os.system('transkun "%s" "%s" --device cuda'%(file,path))
  return path

uploaded = files.upload()
if len(uploaded) >= 2:
    newZip = zipfile.ZipFile("transcribe.zip", 'w')
    for fn in uploaded.keys():
        newZip.write(transcribe(fn, "."), compress_type=zipfile.ZIP_DEFLATED)
    newZip.close()
    files.download("transcribe.zip")
else:
    for fn in uploaded.keys():
      files.download(transcribe(fn, "."))


In [ ]:
!pip install pretty-midi pandas -q

import pretty_midi
import pandas as pd
import os

midi_file = "c-major-scale.mid"

if not os.path.exists(midi_file):
    print("MIDI-файл не знайдено. Перевір назву або завантаж ще раз.")
else:
    midi = pretty_midi.PrettyMIDI(midi_file)

    notes_list = []
    for instrument in midi.instruments:
        print(f"Інструмент: {instrument.name or 'Piano'} (program {instrument.program})")
        for note in instrument.notes:
            notes_list.append({
                'start_sec': round(note.start, 3),
                'end_sec': round(note.end, 3),
                'duration_sec': round(note.end - note.start, 3),
                'midi_pitch': note.pitch,
                'note_name': pretty_midi.note_number_to_name(note.pitch),
                'velocity': note.velocity
            })

    df = pd.DataFrame(notes_list)

    print("\nСписок усіх нот у MIDI:")
    display(df)  # гарна таблиця в Colab

    print("\nУнікальні ноти:")
    print(df['note_name'].unique())

    print(f"\nЗагальна кількість нот: {len(df)}")
    print(f"Тривалість треку: {midi.get_end_time():.2f} сек")

    # Якщо хочеш графік нот (час по X, висота по Y)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 5))
    plt.scatter(df['start_sec'], df['midi_pitch'], s=40, c=df['velocity'], cmap='viridis', alpha=0.7)
    plt.colorbar(label='Velocity (гучність)')
    plt.xlabel('Час (секунди)')
    plt.ylabel('MIDI pitch (нота)')
    plt.title('Візуалізація нот з транскрибованої гами До мажор')
    plt.grid(True)
    plt.show()

Порівняння з класичним CRF — CRF працює по одному кадру, semi-CRF — по цілих сегментах.
Порівняння з CNN — CNN дає локальні ймовірності, semi-CRF додає логіку послідовності та тривалості.